In [13]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [14]:
#create session

spark = (( SparkSession
            .builder
                .appName('purchase_orderline')
                .config('spark.jars', f'{sqlserverjar}')
                .config('spark.driver.extraClassPath', f'{sqlserverjar}')
                .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
        ))

In [15]:
tmp_purchaseorderline = (( spark
                            .read
                            .format('jdbc')
                            .option('url', mssqlsource)
                            .option('driver', mssqldriver)
                            .option('user', mssqluser)
                            .option('password', mssqlpwd)
                            .option('query', 
                                    
                                    """ 
                                        select 
                                            purchaseorderlineid,
                                            purchaseorderid,
                                            stockitemid,
                                            receivedouters,
                                            orderedouters,
                                            packagetypeid,
                                            description,
                                            expectedunitpriceperouter,
                                            cast(isorderlinefinalized as varchar(10)) as isorderlinefinalized,
                                            lastreceiptdate
                                        
                                        from purchasing.purchaseorderlines
                                    
                                    """                            
                                    )
                        )).load()


In [16]:
orderframe = (( tmp_purchaseorderline
                    .withColumnRenamed('purchaseorderlineid', 'orderlineid')
                    .withColumnRenamed('expectedunitpriceperouter', 'expectectedunit')
                    .withColumnRenamed('receivedouters', 'qtdreceived')
                    .withColumnRenamed('orderedouters', 'qtdordered')

            ))

In [17]:
#repalace true-false for 0-1
#reaming column
#drop column

purchaseorder = ((orderframe
                    .select('*', 
                                f.when(orderframe.isorderlinefinalized == 'true', 1)
                                .otherwise('0')
                                    .alias('isfinalized'))
                    
                    #droping old column isorderlinefinalized
                    .drop('isorderlinefinalized')
                )) 

In [18]:
(( 
    purchaseorder
        .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('driver', mssqldriver)
        .option('dbtable', 'fact.purchaseorder')
        .save(mode= 'overwrite')
    ))